<a href="https://colab.research.google.com/github/iamanantalok/Master-Thesis-DES-LCA/blob/main/learning-logs/02_single_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install simpy

In [3]:
import simpy

ARRIVAL_GAP = 3        #minutes between arriving parts
PROCESS_TIME = 5       #minutes it takes to process a part
SIM_TIME = 60          #minutes to simulate

def part(env, name, machine, log):
  arrived = env.now
  with machine.request() as req:
    yield req                     #wait in teh queue for the machine
    started = env.now
    yield env.timeout(PROCESS_TIME)     #being process
    log.append((name, arrived, started, env.now))

def source(env, machine, log):
  i = 0
  while True:
    i+=1
    env.process(part(env, f"part{i}", machine, log))
    yield env.timeout(ARRIVAL_GAP)

env = simpy.Environment()
machine = simpy.Resource(env, capacity=1)
log = []

env.process(source(env, machine, log))
env.run(until=SIM_TIME)


print("name, arrived, started, finished")
for row in log:
  print(row)
print("still waiting in queue:", (len(machine.queue)))


name, arrived, started, finished
('part1', 0, 0, 5)
('part2', 3, 5, 10)
('part3', 6, 10, 15)
('part4', 9, 15, 20)
('part5', 12, 20, 25)
('part6', 15, 25, 30)
('part7', 18, 30, 35)
('part8', 21, 35, 40)
('part9', 24, 40, 45)
('part10', 27, 45, 50)
('part11', 30, 50, 55)
still waiting in queue: 8


#### Adding randomness to above process

In [4]:
import random
import simpy

MEAN_ARRIVAL_GAP = 6  # minutes between arriving parts, on average
MEAN_PROCESS_TIME = 5 # minutes it takes to process a part, on average
SIM_TIME = 480  # one 8-hour shift
SEED = 42


def part(env, name, machine, log):
  arrive = env.now
  with machine.request() as req:
    yield req
    started = env.now
    yield env.timeout(random.expovariate(1.0/MEAN_PROCESS_TIME))
    log.append((name, arrive, started, env.now))

def source(env, machine, log):
  i = 0
  while True:
    i+=1
    env.process(part(env, f"part{i}", machine, log))
    yield env.timeout(random.expovariate(1.0/MEAN_ARRIVAL_GAP))

def run(seed):
  random.seed(seed)
  env = simpy.Environment()
  machine = simpy.Resource(env, capacity=1)
  log = []

  env.process(source(env, machine, log))
  env.run(until=SIM_TIME)

  waits = [started - arrived for (_, arrived, started, _) in log]
  busy = sum(finished - started for (_, _, started, finished) in log)
  return len(log), sum(waits) / len(waits), busy / SIM_TIME


finished, avg_wait, utilisation = run(SEED)
print("parts finished:", finished)
print("average wait (min):", round(avg_wait, 1))
print("machine utilisation:", round(utilisation, 2))

parts finished: 66
average wait (min): 3.3
machine utilisation: 0.59


In [5]:
for seed in range(1, 6):
    f, w, u = run(seed)
    print(seed, f, round(w, 1), round(u, 2))

1 85 37.2 0.88
2 72 7.2 0.77
3 62 7.6 0.68
4 78 30.3 0.85
5 70 18.3 0.86


#### Bottles arrive --> filler --> buffer --> capper --> finished

In [6]:
import random
import simpy

MEAN_ARRIVAL_GAP = 6  # minutes between arriving parts, on average
MEAN_FILL_TIME = 4 # minutes it takes to process a part, on average
MEAN_CAP_TIME = 5 # minutes, on average
BUFFER_SIZE = 3 # minutes
SIM_TIME = 480  # one 8-hour shift


def source(env, inbound):
  i = 0
  while True:
    i += 1
    yield inbound.put(f"bottel{i}")
    yield env.timeout(random.expovariate(1.0/MEAN_ARRIVAL_GAP))

def machine(env, name, mean_time, store_in, store_out, busy):
  while True:
    item = yield store_in.get()
    start = env.now
    yield env.timeout(random.expovariate(1.0/mean_time))
    busy[name] += env.now - start
    yield store_out.put(item)

def run(seed, buffer_size=BUFFER_SIZE):
    random.seed(seed)
    env = simpy.Environment()
    inbound = simpy.Store(env)
    buffer = simpy.Store(env, capacity=buffer_size)
    finished = simpy.Store(env)
    busy = {"filler": 0, "capper": 0}
    env.process(source(env, inbound))
    env.process(machine(env, "filler", MEAN_FILL_TIME, inbound, buffer, busy))
    env.process(machine(env, "capper", MEAN_CAP_TIME, buffer, finished, busy))
    env.run(until=SIM_TIME)
    return len(finished.items), len(inbound.items), busy["filler"] / SIM_TIME, busy["capper"] / SIM_TIME

done, waiting, u_fill, u_cap = run(42)
print("bottles finished:", done)
print("bottles waiting before the filler:", waiting)
print("filler utilisation:", round(u_fill, 2))
print("capper utilisation:", round(u_cap, 2))

bottles finished: 68
bottles waiting before the filler: 11
filler utilisation: 0.54
capper utilisation: 0.91


In [7]:
# import simpy

# def source(env, inbound):
#     for i in range(1, 6):                      # only 5 bottles
#         yield inbound.put(f"bottle{i}")
#         print(f"{env.now:>2}  source   put {f'bottle{i}'} on inbound shelf")
#         yield env.timeout(2)                   # a new bottle every 2 minutes

# def machine(env, name, work_time, store_in, store_out):
#     while True:
#         print(f"{env.now:>2}  {name:<8} waiting for a bottle")
#         item = yield store_in.get()
#         print(f"{env.now:>2}  {name:<8} took {item}, starts working")
#         yield env.timeout(work_time)
#         print(f"{env.now:>2}  {name:<8} finished {item}, tries to pass it on")
#         yield store_out.put(item)
#         print(f"{env.now:>2}  {name:<8} passed {item} on")

# env = simpy.Environment()
# inbound = simpy.Store(env)
# buffer = simpy.Store(env, capacity=1)          # room for only ONE bottle
# finished = simpy.Store(env)

# env.process(source(env, inbound))
# env.process(machine(env, "filler", 2, inbound, buffer))    # fast: 2 minutes
# env.process(machine(env, "capper", 5, buffer, finished))   # slow: 5 minutes
# env.run(until=16)

# print("inbound shelf:", inbound.items)
# print("buffer shelf: ", buffer.items)
# print("finished:     ", finished.items)

#### Adding breakdowns: Real machines fail, and while one is being repaired it produces nothing, but the line is still running around it.

In [10]:
import random
import simpy

MEAN_ARRIVAL_GAP = 6
MEAN_FILL_TIME = 4
MEAN_CAP_TIME = 5
BUFFER_SIZE = 3
MEAN_TIME_TO_FAILURE = 60   # on average, a machine breaks after 60 minutes of running
REPAIR_TIME = 10            # minutes to repair a machine
SIM_TIME = 480


def source(env, inbound):
    i = 0
    while True:
        i += 1
        yield inbound.put(f"bottle{i}")
        yield env.timeout(random.expovariate(1 / MEAN_ARRIVAL_GAP))

def machine(env, name, mean_time, store_in, store_out, stats):
    while True:
        item = yield store_in.get()
        work_left = random.expovariate(1 / mean_time)
        while work_left > 0:
            stats["working"] = True
            start = env.now
            try:
                yield env.timeout(work_left)               # work on the bottle
                stats["busy"] += env.now - start
                work_left = 0                              # done
            except simpy.Interrupt:                        # the machine broke!
                stats["busy"] += env.now - start
                work_left -= env.now - start               # remember the work still to do
                stats["working"] = False
                stats["breakdowns"] += 1
                yield env.timeout(REPAIR_TIME)             # wait for the repair
                stats["down"] += REPAIR_TIME
        stats["working"] = False
        yield store_out.put(item)

def breakdowns(env, machine_process, stats):
    while True:
        yield env.timeout(random.expovariate(1 / MEAN_TIME_TO_FAILURE))
        if stats["working"]:                               # only a running machine can break
            machine_process.interrupt()

def run(seed):
    random.seed(seed)
    env = simpy.Environment()
    inbound = simpy.Store(env)
    buffer = simpy.Store(env, capacity=BUFFER_SIZE)
    finished = simpy.Store(env)
    stats = {
        "filler": {"working": False, "busy": 0, "down": 0, "breakdowns": 0},
        "capper": {"working": False, "busy": 0, "down": 0, "breakdowns": 0},
    }
    env.process(source(env, inbound))
    filler = env.process(machine(env, "filler", MEAN_FILL_TIME, inbound, buffer, stats["filler"]))
    capper = env.process(machine(env, "capper", MEAN_CAP_TIME, buffer, finished, stats["capper"]))
    env.process(breakdowns(env, filler, stats["filler"]))
    env.process(breakdowns(env, capper, stats["capper"]))
    env.run(until=SIM_TIME)
    return len(finished.items), stats

done, stats = run(42)
print("bottles finished:", done)
for name, s in stats.items():
    print(name, "| breakdowns:", s["breakdowns"],
          "| busy:", round(s["busy"] / SIM_TIME, 2),
          "| down:", round(s["down"] / SIM_TIME, 2))

bottles finished: 62
filler | breakdowns: 4 | busy: 0.46 | down: 0.08
capper | breakdowns: 6 | busy: 0.6 | down: 0.12


In [11]:
import random
import simpy
import pandas as pd

MEAN_ARRIVAL_GAP = 6
MEAN_FILL_TIME = 4
MEAN_CAP_TIME = 5
BUFFER_SIZE = 3
MEAN_TIME_TO_FAILURE = 60
REPAIR_TIME = 10
SIM_TIME = 480

# Power draw per state in kW (made-up example values)
POWER_KW = {
    "filler": {"idle": 0.5, "processing": 2.0, "blocked": 0.5, "down": 0.1},
    "capper": {"idle": 0.3, "processing": 1.5, "blocked": 0.3, "down": 0.1},
}

def change_state(env, name, m, new_state, log):
    # close the state the machine was in, and open the new one
    log.append({"machine": name, "state": m["state"], "start": m["since"], "end": env.now})
    m["state"] = new_state
    m["since"] = env.now

def source(env, inbound):
    i = 0
    while True:
        i += 1
        yield inbound.put(f"bottle{i}")
        yield env.timeout(random.expovariate(1 / MEAN_ARRIVAL_GAP))

def machine(env, name, mean_time, store_in, store_out, m, log):
    while True:
        change_state(env, name, m, "idle", log)
        item = yield store_in.get()
        work_left = random.expovariate(1 / mean_time)
        while work_left > 0:
            change_state(env, name, m, "processing", log)
            start = env.now
            try:
                yield env.timeout(work_left)
                work_left = 0
            except simpy.Interrupt:
                work_left -= env.now - start
                change_state(env, name, m, "down", log)
                yield env.timeout(REPAIR_TIME)
        change_state(env, name, m, "blocked", log)
        yield store_out.put(item)

def breakdowns(env, machine_process, m):
    while True:
        yield env.timeout(random.expovariate(1 / MEAN_TIME_TO_FAILURE))
        if m["state"] == "processing":
            machine_process.interrupt()

def run(seed):
    random.seed(seed)
    env = simpy.Environment()
    inbound = simpy.Store(env)
    buffer = simpy.Store(env, capacity=BUFFER_SIZE)
    finished = simpy.Store(env)
    log = []
    machines = {"filler": {"state": "idle", "since": 0},
                "capper": {"state": "idle", "since": 0}}
    env.process(source(env, inbound))
    filler = env.process(machine(env, "filler", MEAN_FILL_TIME, inbound, buffer, machines["filler"], log))
    capper = env.process(machine(env, "capper", MEAN_CAP_TIME, buffer, finished, machines["capper"], log))
    env.process(breakdowns(env, filler, machines["filler"]))
    env.process(breakdowns(env, capper, machines["capper"]))
    env.run(until=SIM_TIME)
    for name, m in machines.items():                     # close whatever state each machine ended in
        change_state(env, name, m, "end", log)
    df = pd.DataFrame(log)
    df["minutes"] = df["end"] - df["start"]
    df = df[df["minutes"] > 0]                           # drop zero-length entries
    return len(finished.items), df

done, log = run(42)
print("bottles finished:", done)
log.head(10)

bottles finished: 62


,machine,state,start,end,minutes
3,filler,processing,0.000000,1.010345,1.010345
5,capper,idle,0.000000,1.010345,1.010345
6,capper,processing,1.010345,1.519730,0.509386
7,filler,idle,1.010345,1.929744,0.919400
8,filler,processing,1.929744,2.293554,0.363810
10,capper,down,1.519730,11.519730,10.000000
11,filler,idle,2.293554,15.293473,12.999919
12,filler,processing,15.293473,15.414474,0.121001
14,capper,processing,11.519730,17.678308,6.158578
17,filler,idle,15.414474,18.581750,3.167276


In [12]:
summary = log.pivot_table(index="machine", columns="state", values="minutes", aggfunc="sum", fill_value=0)
summary.round(1)

state,blocked,down,idle,processing
machine,,,,
capper,0.0,60.0,132.6,287.4
filler,32.2,40.0,188.8,219.0


In [13]:
log["kW"] = log.apply(lambda r: POWER_KW[r["machine"]][r["state"]], axis=1)
log["kWh"] = log["kW"] * log["minutes"] / 60
energy = log.pivot_table(index="machine", columns="state", values="kWh", aggfunc="sum", fill_value=0)
energy["total"] = energy.sum(axis=1)
print(energy.round(2))
print("kWh per bottle:", round(log["kWh"].sum() / done, 3))

state    blocked  down  idle  processing  total
machine                                        
capper      0.00  0.10  0.66        7.19   7.95
filler      0.27  0.07  1.57        7.30   9.21
kWh per bottle: 0.277


In [14]:
log.to_csv("state_log.csv", index=False)